# nano Beatrix — the find screen's follow-up: Alhena-F with the index term at a quarter of its weight

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_find_followup.ipynb)

**Why**: at weight 1 the head's term at the index entries cost the core its copying (held-out text about .32 bits worse on both seeds; the frame copy and, on seed 1, the answers lost) while the find stayed at .86. The plan's rule for that outcome (written before any run): before any change of design, the same core again with the term's weight at .25, so the weight's share of the cost is measured apart from the form's.

| part | what |
|---|---|
| Alhena-F .25 s1, s2 | the find cores (`nano_find_cores.ipynb`) verbatim at 24,000 steps with `head_lam_i=0.25`: L = (1/N) sum_t CE + 1 x (1/Q) sum_q BCE(head at the question's SEP) + .25 x (1/E) sum_e BCE(head at the index entry's SEP) |

They are graded with the cores' own marks (text, frames, the find, the bytes, the answers through the find) against the scale test's stored Alhena of the same seed; no member trains on them here.

**The cells**: C1 pins the trainer and holds the decision block; C2 builds the data, runs 200 steps on this card for the price line (a projection past 1.5x the planned run stops the session for a re-plan) and probes the push path; C3 trains the two cores, one at a time, each shipped as it lands to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) (`bed/reports/`, `bed/weights/<run>/`); C4 renders the table; C5 reads where the find fails on every Alhena-F core on the Hub (by the entry's place in the index, by the record's slot, by how many keys share the asked key's cue word; `bed/reports/find_breakdown/`).

**Before C1**: the notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. A file named `STOP` in `/content/out` ends the session between runs; a restarted session resumes where it stopped.

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.72"
import torch
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| torch', torch.__version__, '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 72), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the corpus, the records, the weights, runner.log
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # every finished run ships here; tokenless = nothing ships
CORPUS_BYTES = 160e6                                   # fineweb-edu text for the bed (the scale test's corpus)
BED = dict(                                            # the scale test's Alhena recipe at 24,000 steps, verbatim
    ctx=4096, mb=8, steps=24000,                       # 24,000 steps x 8 rows x 4,096 bytes; pure Adam, lr 1e-3, 100 warm-up steps, the cosine to a tenth
    frames='dyn', plant=1.0,
    sizes='32,128,256,512', frame_records=4, nest=0.3, overlap=0.0, ask='any', register='caption', tags='mode',
    log_every=100,
)
GRID_ROWS = dict(grid='ag', grid_p=0.5,                # half of every batch's rows are addressed grid rows
                 head='digit', head_lam=1.0,           # the per-digit address head at the questions' SEPs (the scale test's term)
                 trace_every=2000)                     # the formation trace every 2,000 steps
FIND = dict(head_at='both', head_lam_i=0.25)           # THE FOLLOW-UP: the term at each index entry's SEP at a quarter of its weight
PARTS = [                                              # the run tag carries the weight (..._ahdigit1i0.25_find): nothing overwrites the weight-1 cores
    ('Alhena-F .25 s1', dict(BED, **GRID_ROWS, **FIND, arms='alhena-both', seeds='1', series='find')),
    ('Alhena-F .25 s2', dict(BED, **GRID_ROWS, **FIND, arms='alhena-both', seeds='2', series='find')),
]
READ_BATCHES, CALIB_BATCHES = 64, 32                   # the scale test's read and calibration rows (8 rows a batch)
SMOKE_STEPS = 200                                      # C2's short run of Alhena-F on this card for the price line; its record stays local
PLAN_MIN = 16.5                                        # the planned minutes of one 24,000-step Alhena run on this card (978-980 s measured)
STOP_FACTOR = 1.5                                      # a projection past 1.5x the plan stops the session for a re-plan
PARALLEL = 1                                           # ONE run at a time: one 9M craft at 8 x 4,096 saturates the card
print('decision block:', len(PARTS), 'parts | out', OUT, '| ships to', REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the plan, 200 steps of Alhena-F on this card (the price line), the push probe. No part trains.
import json, os, time, torch
from geolip.nano_beatrix import bed_corpus, bed_runner
if not os.path.exists(f'{OUT}/bed/bed_corpus.bin'):
    bed_corpus.build(f'{OUT}/bed', CORPUS_BYTES)
HELD = f'{OUT}/bed/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{OUT}/bed/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(HELD))
todo = 0
for name, spec in PARTS:
    p = bed_runner.run(spec, out=OUT, dry=True)
    todo += p['runs'] - p['done_before']
smoke = dict(PARTS[0][1], steps=SMOKE_STEPS, log_every=20, trace_every=100, read_batches=4, calib_batches=2)
res = bed_runner.run(smoke, out=OUT)
assert res['failed'] is None, 'the short run failed: read the lines above before any part'
last = [s for s in (bed_runner.parse_step(l) for l in open(f'{OUT}/runner.log', encoding='utf-8')) if s][-1]
per_run = BED['steps'] * BED['ctx'] * BED['mb'] / (last['tok_s'] * 1e3) / 60
PRICE_OK = per_run <= STOP_FACTOR * PLAN_MIN
print(f"this card: {last['tok_s']}k tok/s at {BED['ctx']} x {BED['mb']} -> one {BED['steps']:,}-step run about {per_run:.1f} min (planned "
      f"{PLAN_MIN} min); the {todo} runs left about {per_run * todo / 60:.1f} h (+ a few minutes of end reads a run)"
      + ('' if PRICE_OK else f' -> PAST {STOP_FACTOR}x THE PLAN: C3 will not start; re-plan first'))
PUSH_OK = False
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_find_fu.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'tok_s': last['tok_s'], 'parts': [n for n, _ in PARTS]}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano find follow-up: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the cells below run and ship nothing')

In [ ]:
# C3 (T) — THE SESSION: the parts in gate order, one run at a time, resume-first (runner_done.txt in OUT); a STOP file in OUT starts no
# new run; every finished run ships to REPO as it lands (one background shipper: the card never waits on the Hub).
from geolip.nano_beatrix import bed_runner
assert PRICE_OK, 'the price line projected past the plan: re-plan before any part trains'
push = REPO if (HF_TOKEN and PUSH_OK) else ''
shipper = bed_runner._Shipper(push, OUT, print) if push else None
if shipper:
    for rec in bed_runner.pending_records(OUT):
        shipper.submit(rec)
done = []
for name, spec in PARTS:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new part starts')
        break
    print(f'=================== {name} ===================')
    res = bed_runner.run(spec, out=OUT, push=push, parallel=PARALLEL, shipper=shipper)
    done.append((name, res))
    if res['failed'] is not None or res['stopped']:
        print('the part did not finish:', name, {k: v for k, v in res.items() if k != 'ran'})
        break
if shipper:
    shipper.drain(print)
    left = [os.path.basename(p)[4:-5] for p in bed_runner.pending_records(OUT)]
    print('UNSHIPPED:', left if left else 'none')
print('session:', [(n, r['failed'], r['stopped'], len(r['ran'])) for n, r in done])

In [ ]:
# C4 — THE TABLE. The cores of this session: held-out bits, the frame copy, the index (teacher-forced and free-running), the find (the head at the index
# entries on the free-running row), the answers by request, the terms' descent.
# Run any time.
import glob, json
from IPython.display import Markdown, display
rows = ['| run | bits | copy 32 / 128 / 256 / 512 | index TF / FR | find right / truth / Q999 | answers true / via index / via find | descent q / i | min |',
        '|---|---|---|---|---|---|---|---|']
for p in sorted(glob.glob(f'{OUT}/bed/bed_alhena-both_*_L*_*.json')):
    r = json.load(open(p, encoding='utf-8'))
    if not r.get('grid'):
        continue
    ff = r.get('fetch_by_frame') or {}
    copy = ' / '.join(f"{ff[f'data/{s}/depth0']['full_value']:.3f}" if f'data/{s}/depth0' in ff else '-' for s in (32, 128, 256, 512))
    g, hd = r['grid']['read'], r['grid']['head'] or {}
    fr = g.get('free_running') or {}
    hf = fr.get('headif')
    find = f"{hf['right']:.5f} / {hf['truth']:.4f} / {max(fr['calib']['tau']['0.999']):.2e}" if hf else '-'
    via = f"{g['true']['whole']:.4f} / {fr.get('indexf_whole', float('nan')):.4f} / " + (f"{hf['whole']:.4f}" if hf else '-')
    desc = f"{hd.get('descended')} / {(hd.get('index') or {}).get('descended', '-')}"
    _n = os.path.basename(p)[4:-5]                       # the run's arm, seed and steps, then its head weights and series
    rows.append(f"| {_n.split('_ctx')[0]} {_n.split('gag50_')[-1]} | {r['heldout_bpb']:.4f} | {copy} | {g['true']['index_right']:.4f} / "
                f"{fr.get('indexf_right', float('nan')):.4f} | {find} | {via} | {desc} | {r['train_minutes']:.1f} |")
display(Markdown('\n'.join(rows)))

In [ ]:
# C5 (R) — THE FIND'S BREAKDOWN (a read, no prediction): on every Alhena-F core on the Hub (24,000 and 48,000 steps; index-term weights 1
# and .25), the head at each index entry's SEP on the free-running row (every question at the null anchor), decoded at .5 and by argmax,
# right or wrong against the entry's record; tabulated by the entry's place in the index (0-23), by the record's slot (1-24: the farther
# back from the entry, the lower the slot) and by how many of the row's 24 keys share the asked key's cue word; the teacher-forced row
# (the index as composed) beside it. The cores' own read rows (64 x 8 grid rows a seed, held-out filler); fp32, TF32 off; ~1 min a core.
import json, time
import numpy as np
from huggingface_hub import HfApi, snapshot_download
from geolip.nano_beatrix import grid_rows as gr, rotary_scale as rs
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

def _cue(key):
    return key.rstrip('0123456789')

@torch.no_grad()
def breakdown(core, rows, mb=8):
    per = {'F': [], 'T': []}
    for bi in range(0, len(rows), mb):
        batch = rows[bi:bi + mb]
        fr, _, _ = rs.free_rows(core, batch)
        for rk, rr in (('T', batch), ('F', fr)):
            metas = [m for _, m in rr]
            x = rs.to_x(rr, core.dev)
            core.set_streams(metas, 'null')
            h = core.trunk(x[:, :-1])
            core.clear()
            for j, m in enumerate(metas):
                cues = [_cue(r['key']) for r in m['recs']]
                sep = torch.tensor([e['a_pos'] - 1 for e in m['index']], device=core.dev)
                s = core.head_scores(h[j, sep]).cpu().numpy()                   # (24, 2, 8)
                dec = gr.decode_scores(s)
                for qi, e in enumerate(m['index']):
                    rec = m['recs'][e['rec']]
                    t = gr.digits(rec['addr'])
                    per[rk].append(dict(place=qi, slot=rec['slot'], share=cues.count(_cue(rec['key'])), right=tuple(dec[qi]) == tuple(t),
                                        right_argmax=tuple(int(v) for v in s[qi].argmax(-1)) == tuple(t)))
    out = {}
    for rk, lst in per.items():
        def table(key, bins):
            res = []
            for lo, hi in bins:
                sel = [d for d in lst if lo <= d[key] <= hi]
                if sel:
                    res.append(dict(lo=lo, hi=hi, n=len(sel), right=float(np.mean([d['right'] for d in sel])),
                                    right_argmax=float(np.mean([d['right_argmax'] for d in sel]))))
            return res
        out[rk] = dict(n=len(lst), right=float(np.mean([d['right'] for d in lst])), right_argmax=float(np.mean([d['right_argmax'] for d in lst])),
                       by_place=table('place', [(i, i + 3) for i in range(0, 24, 4)]), by_slot=table('slot', [(i, i + 3) for i in range(1, 25, 4)]),
                       by_share=table('share', [(1, 1), (2, 2), (3, 3), (4, 24)]))
    return out

TAIL_B = '_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50_ahdigit1'
have = set(HfApi(token=HF_TOKEN or False).list_repo_files(REPO))
BD = [f'alhena-both_s{s}_L{st}{TAIL_B}i{w}_find' for w in ('1', '0.25') for s in (1, 2) for st in (24000, 48000)]
BD = [t for t in BD if f'bed/weights/{t}/model.safetensors' in have]
print('the cores read:', BD)
os.makedirs(f'{OUT}/find_breakdown', exist_ok=True)
held = rs.load_heldout(HELD)
t_all = time.time()
for n, tag in enumerate(BD, 1):
    snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/hub', token=HF_TOKEN or False)
    core = rs.Core(f'{OUT}/hub/bed/weights/{tag}', 'cuda')
    seed = int(tag.split('_s')[1].split('_')[0])
    res = breakdown(core, rs.gauge_rows(seed, 'read', READ_BATCHES * 8, held))
    json.dump(dict(core=tag, rows=READ_BATCHES * 8, **res), open(f'{OUT}/find_breakdown/BREAKDOWN_{tag}.json', 'w'), indent=1)
    f = res['F']
    print(f"{n}/{len(BD)} {tag.split('_ctx')[0]} {tag.split('gag50_')[-1]}: the find {f['right']:.4f} (argmax {f['right_argmax']:.4f}) of {f['n']}; "
          f"by place {[round(r['right'], 3) for r in f['by_place']]}; by slot {[round(r['right'], 3) for r in f['by_slot']]}; "
          f"by cue sharing {[(r['lo'], r['n'], round(r['right'], 3)) for r in f['by_share']]}; {(time.time() - t_all) / 60:.1f} min spent, "
          f"{len(BD) - n} cores left")
    del core
    torch.cuda.empty_cache()
if HF_TOKEN and PUSH_OK:
    HfApi(token=HF_TOKEN).upload_folder(folder_path=f'{OUT}/find_breakdown', path_in_repo='bed/reports/find_breakdown', repo_id=REPO,
                                        commit_message='nano find: the find breakdown (a read)')
    print('shipped bed/reports/find_breakdown/')